In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [2]:
#データセットの用意と前処理
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s]


In [3]:
#教師モデルと生徒モデル
class teacher_model(nn.Module):
  def __init__(self, num_classes = 10):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 128, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.Conv2d(128, 64, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(64, 64, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.Conv2d(64, 32, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Linear(2048, 512),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(512, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x



class student_model(nn.Module):
  def __init__(self, num_classes = 10):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(16, 16, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Linear(1024, 256),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(256, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x

In [4]:
#trainとtest関数を作成

def train(model, train_loader, epochs, learning_rate, device):
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr = learning_rate)
  model.train()

  for epoch in range(epochs):
    total_loss = 0.0
    for inputs, labels in train_loader:
      inputs, labels = inputs.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(inputs)

      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

    print(f"Epoch: {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader)}")


def test(model, test_loader, device):
  model.to(device)
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for inputs, labels in test_loader:
      inputs, labels = inputs.to(device), labels.to(device)

      outputs = model(inputs)
      _, pred = torch.max(outputs.data, 1)

      total += labels.size(0)
      correct += (pred == labels).sum().item()

  accuracy = 100 * correct / total
  print(f"Test Accuracy: {accuracy:.2f}%")
  return accuracy

In [5]:
#教師モデルの学習
torch.manual_seed(42)
teacher = teacher_model(num_classes = 10).to(device)
train(teacher, train_loader, epochs=10, learning_rate=0.001, device=device)


Epoch: 1/10, Loss: 1.3310228220337188
Epoch: 2/10, Loss: 0.86211748897572
Epoch: 3/10, Loss: 0.6800540302263196
Epoch: 4/10, Loss: 0.533724022490899
Epoch: 5/10, Loss: 0.4209327744248578
Epoch: 6/10, Loss: 0.31631616691646675
Epoch: 7/10, Loss: 0.23497372021531815
Epoch: 8/10, Loss: 0.17497469097032875
Epoch: 9/10, Loss: 0.14523500374153905
Epoch: 10/10, Loss: 0.12077240700192769


In [6]:
#生徒モデルの学習
torch.manual_seed(42)
student = student_model(num_classes = 10).to(device)
train(student, train_loader, epochs=10, learning_rate=0.001, device=device)


Epoch: 1/10, Loss: 1.4673611785444762
Epoch: 2/10, Loss: 1.1526381626458424
Epoch: 3/10, Loss: 1.0221264161112364
Epoch: 4/10, Loss: 0.9212399612912132
Epoch: 5/10, Loss: 0.8445586950882621
Epoch: 6/10, Loss: 0.7788522374599486
Epoch: 7/10, Loss: 0.7148051639956892
Epoch: 8/10, Loss: 0.6529319963949111
Epoch: 9/10, Loss: 0.6006788024512093
Epoch: 10/10, Loss: 0.5524516700936095


In [7]:
#2つのモデルのパラメータ数の確認
total_params_teacher = sum(p.numel() for p in teacher.parameters())
print(f"teacher_model parameters: {total_params_teacher}")

total_params_student = sum(p.numel() for p in student.parameters())
print(f"student_model parameters: {total_params_student}")


teacher_model parameters: 1186986
student_model parameters: 267738


In [8]:
#2つのモデルのテスト結果
test_accuracy_teacher = test(teacher, test_loader, device)
test_accuracy_student = test(student, test_loader, device)

print(f"teacher_model_accuracy: {test_accuracy_teacher:.2f}%")
print(f"student_model_accuracy: {test_accuracy_student:.2f}%")

Test Accuracy: 75.25%
Test Accuracy: 70.49%
teacher_model_accuracy: 75.25%
student_model_accuracy: 70.49%


In [9]:
#知識蒸留の実装
def knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
  ce_loss = nn.CrossEntropyLoss()
  optimizer = optim.Adam(student.parameters(), lr=learning_rate)

  teacher.eval()
  student.train()

  for epoch in range(epochs):
    total_loss = 0.0
    for inputs, labels in train_loader:
      inputs, labels = inputs.to(device), labels.to(device)

      optimizer.zero_grad()

      with torch.no_grad():
        teacher_logits = teacher(inputs)

      student_logits = student(inputs)

      soft_targets = nn.functional.softmax(teacher_logits / T, dim = -1) #temperatureで平滑化した教師モデルのソフトターゲット
      soft_prob = nn.functional.log_softmax(student_logits / T, dim = -1) #生徒モデルのソフトターゲットを対数変換

      soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob) / soft_prob.size()[0] * (T ** 2))
      label_loss = ce_loss(student_logits, labels)

      loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss

      loss.backward()
      optimizer.step()

      total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(train_loader)}")


In [10]:
#蒸留とtest

knowledge_distillation(teacher, student, train_loader, epochs = 10, learning_rate = 0.001,
                       T = 2, soft_target_loss_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_ce_kd_student = test(student, test_loader, device)


print(f"teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"student accuracy without teacher: {test_accuracy_student:.2f}%")
print(f"student accuracy with CE + KD: {test_accuracy_ce_kd_student:.2f}%")

Epoch 1/10, Loss: 0.8223741652105775
Epoch 2/10, Loss: 0.7404439530104322
Epoch 3/10, Loss: 0.6831897770047493
Epoch 4/10, Loss: 0.6394588222436588
Epoch 5/10, Loss: 0.5922312920203294
Epoch 6/10, Loss: 0.5488916559292533
Epoch 7/10, Loss: 0.5037613998136252
Epoch 8/10, Loss: 0.4720274722942001
Epoch 9/10, Loss: 0.4529927338633086
Epoch 10/10, Loss: 0.41819923506368456
Test Accuracy: 69.69%
teacher accuracy: 75.25%
student accuracy without teacher: 70.49%
student accuracy with CE + KD: 69.69%


# **Cosine**

In [11]:
#隠れ状態の出力を追加した教師モデルと生徒モデル

class Modified_teacher_cosine(nn.Module):
  def __init__(self, num_classes = 10):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 128, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.Conv2d(128, 64, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(64, 64, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.Conv2d(64, 32, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Linear(2048, 512),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(512, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    flattened_conv_output = torch.flatten(x, 1) #隠れ層の特徴を出力
    x = self.classifier(flattened_conv_output)
    flattened_conv_output_after_pooling = torch.nn.functional.avg_pool1d(flattened_conv_output, 2)
    return x, flattened_conv_output_after_pooling


class Modified_student_cosine(nn.Module):
  def __init__(self, num_classes = 10):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(16, 16, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Linear(1024, 256),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(256, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    flattened_conv_output = torch.flatten(x, 1)  #隠れ層の特徴を出力
    x = self.classifier(flattened_conv_output)
    return x, flattened_conv_output

In [12]:
#学習済みの教師モデルの重みコピーと生徒モデルの実装

modified_teacher = Modified_teacher_cosine(num_classes=10).to(device)
modified_teacher.load_state_dict(teacher.state_dict())

torch.manual_seed(42)
modified_student = Modified_student_cosine(num_classes=10).to(device)

In [13]:
#cosine類似度に基づくlossを追加した知識蒸留の実装

def train_cosine_loss(teacher, student, train_loader, epochs, learning_rate, hidden_rep_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.to(device)
    student.to(device)
    teacher.eval()
    student.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                _, teacher_hidden_representation = teacher(inputs)

            student_logits, student_hidden_representation = student(inputs)

            hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(device)) #隠れ層の特徴量についてcosine類似度を計算
            label_loss = ce_loss(student_logits, labels)

            loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [14]:
#隠れ状態を無視したtestの実装

def test_multiple_outputs(model, test_loader, device):
  model.to(device)
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for inputs, labels in test_loader:
      inputs, labels = inputs.to(device), labels.to(device)
      outputs, _ = model(inputs)
      _, predicted = torch.max(outputs.data, 1)

      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  accuracy = 100 * correct / total
  print(f"Test Accuracy: {accuracy:.2f}%")
  return accuracy

In [15]:
#蒸留とtest

train_cosine_loss(modified_teacher, modified_student, train_loader, epochs=10, learning_rate=0.001,
                  hidden_rep_loss_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_student_ce_and_cosine_loss = test_multiple_outputs(modified_student, test_loader, device)

Epoch 1/10, Loss: 1.2916501405293985
Epoch 2/10, Loss: 1.0625422500893282
Epoch 3/10, Loss: 0.9690622844354576
Epoch 4/10, Loss: 0.8992005388450135
Epoch 5/10, Loss: 0.8453713794193609
Epoch 6/10, Loss: 0.7987351716326936
Epoch 7/10, Loss: 0.7542655089932024
Epoch 8/10, Loss: 0.7175880231515831
Epoch 9/10, Loss: 0.6818289965619821
Epoch 10/10, Loss: 0.6503074465657744
Test Accuracy: 70.99%


# **inter mediate regressor**

In [16]:
#教師と生徒の畳み込み特徴量を抽出
sample_input = torch.randn(128, 3, 32, 32).to(device)

convolutional_fe_output_teacher = teacher.features(sample_input)
convolutional_fe_output_student = student.features(sample_input)

print("Teacher's feature extractor output shape: ", convolutional_fe_output_teacher.shape)
print("Student's feature extractor output shape: ", convolutional_fe_output_student.shape)


Teacher's feature extractor output shape:  torch.Size([128, 32, 8, 8])
Student's feature extractor output shape:  torch.Size([128, 16, 8, 8])


In [17]:
#中間層の特徴量学習を可能にするよう調整された教師モデルと生徒モデル

class Modified_teacher_regressor(nn.Module):
  def __init__(self, num_classes = 10):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 128, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.Conv2d(128, 64, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(64, 64, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.Conv2d(64, 32, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Linear(2048, 512),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(512, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    conv_feature_map = x #中間層の特徴量を出力
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x, conv_feature_map


class Modified_student_regressor(nn.Module):
  def __init__(self, num_classes = 10):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(16, 16, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2)
    )

    self.regressor = nn.Sequential(
        nn.Conv2d(16, 32, kernel_size = 3, padding = 1)
    )

    self.classifier = nn.Sequential(
        nn.Linear(1024, 256),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(256, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    regressor_output = self.regressor(x) #生徒モデルの中間層の特徴量を教師モデルのサイズ・チャネル数に近づける
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x, regressor_output

In [18]:
#中間層の特徴量での知識蒸留

def train_regressor_ce_loss(teacher, student, train_loader, epochs, learning_rate, feature_map_weight, ce_loss_weight, device):
  ce_loss = nn.CrossEntropyLoss()
  mse_loss = nn.MSELoss()
  optimizer = optim.Adam(student.parameters(), lr=learning_rate)

  teacher.to(device)
  student.to(device)
  teacher.eval()
  student.train()

  for epoch in range(epochs):
    total_loss = 0.0
    for inputs, labels in train_loader:
      inputs, labels = inputs.to(device), labels.to(device)
      optimizer.zero_grad()

      with torch.no_grad():
        _, teacher_feature_map = teacher(inputs)

      student_logits, regressor_feature_map = student(inputs)

      hidden_rep_loss = mse_loss(regressor_feature_map, teacher_feature_map) #中間層の特徴量の誤差
      label_loss = ce_loss(student_logits, labels)

      loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss

      loss.backward()
      optimizer.step()

      total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(train_loader)}")


torch.manual_seed(42)
modified_student_regressor = Modified_student_regressor(num_classes=10).to(device)

modified_teacher_regressor = Modified_teacher_regressor(num_classes=10).to(device)
modified_teacher_regressor.load_state_dict(teacher.state_dict())

train_regressor_ce_loss(modified_teacher_regressor, modified_student_regressor, train_loader, epochs=10, learning_rate=0.001, feature_map_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_student_ce_and_mse_loss = test_multiple_outputs(modified_student_regressor, test_loader, device)

Epoch 1/10, Loss: 1.7182077102344055
Epoch 2/10, Loss: 1.3379984542232035
Epoch 3/10, Loss: 1.1936987135416406
Epoch 4/10, Loss: 1.098801125498379
Epoch 5/10, Loss: 1.0217899494158946
Epoch 6/10, Loss: 0.9576050849521861
Epoch 7/10, Loss: 0.9027347819274648
Epoch 8/10, Loss: 0.8530855474569609
Epoch 9/10, Loss: 0.8085394943766582
Epoch 10/10, Loss: 0.7709039318592043
Test Accuracy: 70.93%


In [19]:
#これまでの全ての結果の確認
print(f"Teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_student:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_ce_kd_student:.2f}%")
print(f"Student accuracy with CE + CosineLoss: {test_accuracy_student_ce_and_cosine_loss:.2f}%")
print(f"Student accuracy with CE + RegressorMSE: {test_accuracy_student_ce_and_mse_loss:.2f}%")

Teacher accuracy: 75.25%
Student accuracy without teacher: 70.49%
Student accuracy with CE + KD: 69.69%
Student accuracy with CE + CosineLoss: 70.99%
Student accuracy with CE + RegressorMSE: 70.93%
